In [14]:
from datasets import load_dataset

In [15]:
ds = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="likes.parquet")

In [16]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['uid', 'timestamp', 'item_id', 'is_organic'],
        num_rows: 881456
    })
})


In [17]:
dataset=ds['train']
print(dataset.features)

{'uid': Value('uint32'), 'timestamp': Value('uint32'), 'item_id': Value('uint32'), 'is_organic': Value('uint8')}


In [18]:
dataset.to_parquet("likes_50m.parquet")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

11458928

In [19]:
import polars as pl

In [20]:
lf=pl.scan_parquet("likes_50m.parquet")
metrics_plan=(
    lf.select([
        pl.col("uid").n_unique().alias("n_users"),
        pl.col("item_id").n_unique().alias("n_items"),
        pl.len().alias("n_interactions"),
        (pl.col("is_organic")==1).sum().alias("organic_interactions")
    ])
    .with_columns([
        (1.0-(pl.col("n_interactions")/(pl.col("n_users")*pl.col("n_items")))).alias("sparsity"),
        (pl.col("organic_interactions")/pl.col("n_interactions")).alias("organic_ratio")
    ])
)

print("===Базовые метрики датасета ===")
print(metrics_plan.collect())

===Базовые метрики датасета ===
shape: (1, 6)
┌─────────┬─────────┬────────────────┬──────────────────────┬──────────┬───────────────┐
│ n_users ┆ n_items ┆ n_interactions ┆ organic_interactions ┆ sparsity ┆ organic_ratio │
│ ---     ┆ ---     ┆ ---            ┆ ---                  ┆ ---      ┆ ---           │
│ u32     ┆ u32     ┆ u32            ┆ u32                  ┆ f64      ┆ f64           │
╞═════════╪═════════╪════════════════╪══════════════════════╪══════════╪═══════════════╡
│ 8283    ┆ 181304  ┆ 881456         ┆ 502201               ┆ 0.999413 ┆ 0.56974       │
└─────────┴─────────┴────────────────┴──────────────────────┴──────────┴───────────────┘


In [21]:
import numpy as np

In [22]:
item_stats = (
    lf.group_by("item_id")
    .agg(pl.len().alias("interactions"))
    .sort("interactions", descending=True)
    .collect()
)

def calculate_gini(array: np.ndarray) -> float:
    """Вычисляет индекс Джини для массива значений."""
    array = np.sort(array).astype(np.float64)
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return ((np.sum((2 * index - n  - 1) * array)) / (n * np.sum(array)))

# Переводим колонку в numpy-массив для быстрых математических операций
gini_index = calculate_gini(item_stats["interactions"].to_numpy())

print(f"\n=== Анализ смещения ===")
print(f"Индекс Джини: {gini_index:.4f}")
# Если значение > 0.9 — датасет имеет экстремальный "тяжелый хвост".
# Это значит, что модели придется жестко штрафовать за предсказание популярных треков.


=== Анализ смещения ===
Индекс Джини: 0.6833


In [23]:
# Сортируем данные внутри партиций по каждому пользователю
session_plan = (
    lf.sort(["uid", "timestamp"])
    .with_columns([
        # diff() считает разницу с предыдущей строкой, over() изолирует расчеты внутри каждого uid
        pl.col("timestamp").diff().over("uid").alias("time_delta")
    ])
    .filter(pl.col("time_delta").is_not_null()) # Убираем первый лайк пользователя (ему не с чем сравнивать)
    .select([
        pl.col("time_delta").mean().alias("mean_time_between_likes"),
        pl.col("time_delta").median().alias("median_time_between_likes")
    ])
)

print("\n=== Временные характеристики (секунды) ===")
print(session_plan.collect())


=== Временные характеристики (секунды) ===
shape: (1, 2)
┌─────────────────────────┬───────────────────────────┐
│ mean_time_between_likes ┆ median_time_between_likes │
│ ---                     ┆ ---                       │
│ f64                     ┆ f64                       │
╞═════════════════════════╪═══════════════════════════╡
│ 142149.401041           ┆ 500.0                     │
└─────────────────────────┴───────────────────────────┘
